# Perturb-seq to HiPoNet Point Clouds

This notebook loads the batch-corrected HVG matrix and groups cells into populations.
Each population is one point cloud (cells x genes). If you have 100 populations, you will have 100 point clouds.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import issparse

In [2]:
# Paths
PROJECT_ROOT = Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "subset_top_5000_hvg_batch_rm.h5ad"
OUT_DIR = PROJECT_ROOT / "perturb-seq"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading: {DATA_PATH}")
adata = sc.read_h5ad(DATA_PATH)
print(adata)

Loading: /nfs/roberts/project/pi_sk2433/sv496/HiPoNet/perturb-seq/subset_top_5000_hvg_batch_rm.h5ad
AnnData object with n_obs × n_vars = 214449 × 5000
    obs: 'batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'perturbed_gene', 'perturbation_barcode', 'cell_barcode', 'experiment', 'batch_name'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'batch_colors', 'hvg', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_phate_2D', 'X_phate_2D_PCA', 'X_phate_3D', 'X_u

In [5]:
from pathlib import Path
import anndata as ad

h5ad_path = Path('/home/sv496/project_pi_sk2433/sv496/HiPoNet/perturb-seq/preprocessed_perturb_seq.h5ad')

# On-disk file size
size_bytes = h5ad_path.stat().st_size
size_mb = size_bytes / (1024 ** 2)
size_gb = size_bytes / (1024 ** 3)

# Dataset dimensions without fully loading into memory
adata = ad.read_h5ad(h5ad_path, backed='r')
n_cells, n_genes = adata.shape
adata.file.close()

print(f'Path: {h5ad_path}')
print(f'File size: {size_bytes:,} bytes ({size_mb:.2f} MB, {size_gb:.2f} GB)')
print(f'Dataset shape: {n_cells:,} cells x {n_genes:,} genes')

Path: /home/sv496/project_pi_sk2433/sv496/HiPoNet/perturb-seq/preprocessed_perturb_seq.h5ad
File size: 10,576,851,960 bytes (10086.87 MB, 9.85 GB)
Dataset shape: 214,449 cells x 36,601 genes


In [6]:
adata

AnnData object with n_obs × n_vars = 214449 × 36601 backed at '/home/sv496/project_pi_sk2433/sv496/HiPoNet/perturb-seq/preprocessed_perturb_seq.h5ad'
    obs: 'batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'perturbed_gene', 'perturbation_barcode', 'cell_barcode', 'experiment', 'batch_name'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'batch_colors', 'hvg', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    o

In [3]:
# Inspect metadata columns available for grouping
print("obs columns:")
print(list(adata.obs.columns))

display(adata.obs.head())

obs columns:
['batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'perturbed_gene', 'perturbation_barcode', 'cell_barcode', 'experiment', 'batch_name']


,batch,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,total_counts_mt,...,log1p_total_counts_ribo,pct_counts_ribo,total_counts_hb,log1p_total_counts_hb,pct_counts_hb,perturbed_gene,perturbation_barcode,cell_barcode,experiment,batch_name
JAG1:GATGCGCCCTGCCCGGCGTGC:AAACCCACAATCGCAT-scRNAseq_2kG_11AMDox_1-treatment,0,4528,8.418256,21486.0,9.975204,33.468305,45.773992,55.668808,67.406683,2345.0,...,8.620652,25.802847,0.0,0.0,0.0,JAG1,GATGCGCCCTGCCCGGCGTGC,AAACCCACAATCGCAT,treatment,scRNAseq_2kG_11AMDox_1
AGAP2:GAAGTGAGGTCGACAGACGGA:AAACCCATCTCCGAAA-scRNAseq_2kG_11AMDox_1-treatment,0,3883,8.264621,15760.0,9.665294,31.453046,44.035533,54.517766,66.865482,1431.0,...,8.316789,25.958120,0.0,0.0,0.0,AGAP2,GAAGTGAGGTCGACAGACGGA,AAACCCATCTCCGAAA,treatment,scRNAseq_2kG_11AMDox_1
PCMTD2:GCCGCGGGCCTGGCACTTAAA:AAAGGTAAGGCACAAC-scRNAseq_2kG_11AMDox_1-treatment,0,3037,8.018955,11171.0,9.321166,31.304270,44.624474,55.939486,69.232835,919.0,...,8.063377,28.421806,0.0,0.0,0.0,PCMTD2,GCCGCGGGCCTGGCACTTAAA,AAAGGTAAGGCACAAC,treatment,scRNAseq_2kG_11AMDox_1
PLEKHH3:GCCCCGGGACTGGCGCGCTG:AAAGTCCAGGTGATAT-scRNAseq_2kG_11AMDox_1-treatment,0,3497,8.159947,13190.0,9.487290,31.152388,45.102350,55.799848,67.680061,694.0,...,8.339501,31.728584,0.0,0.0,0.0,PLEKHH3,GCCCCGGGACTGGCGCGCTG,AAAGTCCAGGTGATAT,treatment,scRNAseq_2kG_11AMDox_1
CHCHD10:GCCACCGCCGCCACCATGCCT:AACAAAGCAGCGCGTT-scRNAseq_2kG_11AMDox_1-treatment,0,3789,8.240121,14981.0,9.614605,29.857820,42.360323,52.646686,65.803351,1268.0,...,8.272315,26.119751,0.0,0.0,0.0,CHCHD10,GCCACCGCCGCCACCATGCCT,AACAAAGCAGCGCGTT,treatment,scRNAseq_2kG_11AMDox_1


In [4]:
# Candidate grouping choices
candidate_groupings = [
    ["perturbed_gene"],
    ["perturbed_gene", "batch_name"],
    ["perturbed_gene", "experiment"],
]

for cols in candidate_groupings:
    if all(col in adata.obs.columns for col in cols):
        n_groups = adata.obs.groupby(cols, observed=True).ngroups
        print(f"Grouping {cols} -> {n_groups} populations")
    else:
        print(f"Grouping {cols} skipped (missing columns)")

Grouping ['perturbed_gene'] -> 2620 populations
Grouping ['perturbed_gene', 'batch_name'] -> 49265 populations
Grouping ['perturbed_gene', 'experiment'] -> 5164 populations


In [5]:
# Choose grouping and size controls
GROUP_BY = ["perturbed_gene"]  # change if you want finer populations
MIN_CELLS_PER_POP = 20
MAX_CELLS_PER_POP = None  # e.g. 300 to subsample very large populations
RNG_SEED = 0
rng = np.random.default_rng(RNG_SEED)

In [6]:
def build_populations(adata, group_by, min_cells=20, max_cells=None, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)

    grouped = adata.obs.groupby(group_by, observed=True).indices
    populations = []
    labels = []
    group_names = []
    sizes = []

    for label_idx, (group_key, row_idx) in enumerate(grouped.items()):
        idx = np.asarray(row_idx)
        if idx.size < min_cells:
            continue

        if max_cells is not None and idx.size > max_cells:
            idx = rng.choice(idx, size=max_cells, replace=False)

        Xg = adata.X[idx]
        if issparse(Xg):
            Xg = Xg.toarray()
        Xg = np.asarray(Xg, dtype=np.float32)

        populations.append(Xg)
        labels.append(label_idx)

        if isinstance(group_key, tuple):
            group_name = "__".join(map(str, group_key))
        else:
            group_name = str(group_key)

        group_names.append(group_name)
        sizes.append(Xg.shape[0])

    if len(populations) == 0:
        raise ValueError("No populations survived filtering. Lower MIN_CELLS_PER_POP or adjust GROUP_BY.")

    return populations, np.asarray(labels), np.asarray(group_names, dtype=object), np.asarray(sizes)

In [7]:
populations, labels, group_names, pop_sizes = build_populations(
    adata=adata,
    group_by=GROUP_BY,
    min_cells=MIN_CELLS_PER_POP,
    max_cells=MAX_CELLS_PER_POP,
    rng=rng,
)

print(f"Number of populations (point clouds): {len(populations)}")
print(f"Gene dimension per point: {populations[0].shape[1]}")
print(f"Population size stats: min={pop_sizes.min()}, median={np.median(pop_sizes):.1f}, max={pop_sizes.max()}")

preview = pd.DataFrame({
    "group_name": group_names[:10],
    "n_cells": pop_sizes[:10],
})
display(preview)

Number of populations (point clouds): 2558
Gene dimension per point: 5000
Population size stats: min=20, median=83.0, max=3284


,group_name,n_cells
0,9p21-add-AL354709-1,21
1,9p21-add-AL359922-2,74
2,9p21-add-AL449423-1,73
3,9p21-add-CDKN2A-alt-TSS,68
4,AACS,86
5,AAGAB,103
6,AAMP,69
7,ABCA5,120
8,ABCB8,77
9,ABCC1,91


In [8]:
# Save in a memory-friendlier streamed pickle format
import pickle

group_keys = np.asarray(GROUP_BY, dtype=object)
cache_path = OUT_DIR / "perturbseq_populations_hvg5000_batch_rm.pkl"

payload = {
    "populations": populations,
    "labels": labels,
    "num_labels": np.unique(labels).size,
    "group_keys": group_keys,
    "group_names": group_names,
}

with open(cache_path, "wb") as handle:
    pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved population cache: {cache_path}")

Saved population cache: /nfs/roberts/project/pi_sk2433/sv496/HiPoNet/perturb-seq/perturb-seq/perturbseq_populations_hvg5000_batch_rm.pkl


In [2]:
# Optional: quick sanity check by reloading the cache
import pickle
cache_path = "perturb-seq/perturbseq_populations_hvg5000_batch_rm.pkl"
with open(cache_path, "rb") as handle:
    loaded = pickle.load(handle)

print("keys:", list(loaded.keys()))
print("n point clouds:", len(loaded["populations"]))
print("first cloud shape:", loaded["populations"][0].shape)
print("group keys:", loaded["group_keys"])

keys: ['populations', 'labels', 'num_labels', 'group_keys', 'group_names']
n point clouds: 2558
first cloud shape: (21, 5000)
group keys: ['perturbed_gene']


## Next step: train unsupervised HiPoNet

Example command from project root:

uv run python unsupervised_main.py --raw_dir perturb-seq/perturbseq_populations_hvg5000_batch_rm.pkl --save_dir checkpoints/perturbseq_hvg5000 --latent_dim 32 --num_epochs 120 --batch_size 8 --dist_weight 0.1 --phate_color_by perturbed_gene --disable_wb